# Ancestry filtering: AoU's premade PCs

Filters directly on AoU's own genomic ancestry PCA (`ancestry_preds.tsv`'s `pca_features`, 16 PCs) instead of building a from-scratch 1000G-projection pipeline. Replaces `01_build_1000g_reference.ipynb` / `02_build_ancestry_panel_hm3.ipynb` / `03_round2_1000g_filter.ipynb` entirely -- no ID+REF+ALT harmonization, no dsub/Batch, no variant-level QC. AoU already computed the PC space; we just filter on it.

`training_pca.tsv` (AoU's own reference panel, HGDP+1000G-style, ~4,151 samples) sits in the *same* 16-PC space as `pca_features` -- no projection needed to orient AoU samples relative to reference populations, just plot both on the same axes.

Approach: Mahalanobis distance from each AoU sample to a reference population's centroid (mean + covariance of `training_pca.tsv`'s matching `pop_label` rows), gated at different quantile thresholds. EUR gets three widths (`loose`/`base`/`strict`) plus a density-reweighted uniform resample (`eur_uniform`); AFR and EAS each get one threshold.

## Inputs

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
from scipy.stats import chi2
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
CDR_VERSION = "v9"

# Top-level bucket folder name for this project's outputs -- distinct from
# CDR_VERSION, which keeps its real meaning elsewhere. Fixed literal, matches
# every other notebook in this pipeline.
PROJECT_DIR = "covariance_v9"

ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{PROJECT_DIR}/01_ancestry_filtering"
FINAL_PCA_DIR = f"{ANCESTRY_BUCKET_DIR}/ancestry_pca_filter/final_pca"
os.makedirs(FINAL_PCA_DIR, exist_ok=True)

# confirmed via a real `ls` of the mounted v9 CDR
ANCESTRY_AUX_DIR = os.path.expanduser(
    "~/workspace/cdrv9/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/aux/ancestry"
)
ANCESTRY_PREDS_PATH = os.path.join(ANCESTRY_AUX_DIR, "ancestry_preds.tsv")
TRAINING_PCA_PATH = os.path.join(ANCESTRY_AUX_DIR, "training_pca.tsv")
assert os.path.isfile(ANCESTRY_PREDS_PATH), f"missing {ANCESTRY_PREDS_PATH!r}"
assert os.path.isfile(TRAINING_PCA_PATH), f"missing {TRAINING_PCA_PATH!r}"

N_PCS_TOTAL = 16   # confirmed via eigenvalues.txt (16 rows) and a real pca_features/scores row

print(FINAL_PCA_DIR)
print(ANCESTRY_PREDS_PATH)
print(TRAINING_PCA_PATH)

## Load AoU samples + reference panel

Both `pca_features` and `scores` are Python-literal bracketed lists of 16 floats -- `ast.literal_eval`, no custom parsing needed.

In [ ]:
def parse_pc_column(series, n_pcs=N_PCS_TOTAL):
    parsed = series.apply(ast.literal_eval)
    arr = np.vstack(parsed.values)
    assert arr.shape[1] == n_pcs, f"expected {n_pcs} PCs, got {arr.shape[1]}"
    return pd.DataFrame(arr, columns=[f"PC{i}" for i in range(1, n_pcs + 1)], index=series.index)

aou = pd.read_csv(ANCESTRY_PREDS_PATH, sep="\t")
aou_pcs = parse_pc_column(aou["pca_features"])
aou = pd.concat([aou[["research_id", "ancestry_pred"]], aou_pcs], axis=1)
aou = aou.rename(columns={"research_id": "person_id"})

ref = pd.read_csv(TRAINING_PCA_PATH, sep="\t")
ref_pcs = parse_pc_column(ref["scores"])
ref = pd.concat([ref[["s", "pop_label", "project_meta.project_pop"]], ref_pcs], axis=1)
ref = ref.rename(columns={"s": "sample", "project_meta.project_pop": "project_pop"})

print(f"{len(aou)} AoU samples, {len(ref)} reference samples")
print("AoU ancestry_pred counts:")
print(aou["ancestry_pred"].value_counts())
print("Reference pop_label counts:")
print(ref["pop_label"].value_counts())

## Mahalanobis gate

Same `mahal()`/`chi2.ppf` pattern used throughout this pipeline's earlier ellipsoid fits. Mean, not mode -- reference `pop_label` clusters are expected to be roughly unimodal, and mean gives a closed-form Mahalanobis distance.

In [ ]:
def mahal(x, mean, cov_inv):
    d = x - mean
    return np.sqrt(d @ cov_inv @ d)

def fit_group(ref_df, group, n_pcs):
    pc_cols = [f"PC{i}" for i in range(1, n_pcs + 1)]
    group_pcs = ref_df.loc[ref_df["pop_label"] == group, pc_cols].values
    assert len(group_pcs) > n_pcs, f"too few reference samples ({len(group_pcs)}) for {group!r}"
    mean = group_pcs.mean(axis=0)
    cov_inv = np.linalg.inv(np.cov(group_pcs, rowvar=False))
    return mean, cov_inv

def gate(aou_df, mean, cov_inv, threshold_quantile, n_pcs):
    pc_cols = [f"PC{i}" for i in range(1, n_pcs + 1)]
    threshold = np.sqrt(chi2.ppf(threshold_quantile, df=n_pcs))
    dists = np.array([mahal(row, mean, cov_inv) for row in aou_df[pc_cols].values])
    return dists, dists <= threshold

## Compare PCs 1-5 vs PCs 1-2

Fit each group's centroid/covariance both ways, compare retained counts at the same threshold quantile before committing to one `n_pcs`. Default assumption is 5 -- more PCs should resolve finer sub-structure (e.g. within-EUR), but only if those PCs actually carry population signal rather than noise for these groups.

In [ ]:
GROUPS = ["eur", "afr", "eas"]
COMPARE_THRESHOLD = 0.999   # arbitrary fixed quantile, just for the n_pcs comparison

for n_pcs in (5, 2):
    print(f"--- n_pcs={n_pcs} ---")
    for group in GROUPS:
        mean, cov_inv = fit_group(ref, group, n_pcs)
        _, keep_mask = gate(aou, mean, cov_inv, COMPARE_THRESHOLD, n_pcs)
        print(f"  {group}: {keep_mask.sum()} / {len(aou)} retained at q={COMPARE_THRESHOLD}")

## Pick `N_PCS`

Set after inspecting the comparison above.

In [ ]:
N_PCS = 5   # <-- set after reviewing apf-npcs-compare's output above

## Sample sets

EUR gets three widths; AFR and EAS each get one threshold. `prob_tag` mirrors the earlier pipeline's convention (`f"p{{threshold*100:g}}"`) so downstream notebooks' file-naming pattern is unchanged.

In [ ]:
SAMPLE_SETS = {
    "eur_strict": {"group": "eur", "threshold": 0.99},
    "eur_base":   {"group": "eur", "threshold": 0.999999},
    "eur_loose":  {"group": "eur", "threshold": 0.99999999},
    "afr":        {"group": "afr", "threshold": 0.999},
    "eas":        {"group": "eas", "threshold": 0.999},
}

def prob_tag(threshold):
    return f"p{threshold * 100:g}"

for cfg in SAMPLE_SETS.values():
    cfg["prob_tag"] = prob_tag(cfg["threshold"])

print(SAMPLE_SETS)

## Run the gate, write keep-lists

In [ ]:
keep_masks = {}
group_fits = {group: fit_group(ref, group, N_PCS) for group in GROUPS}

for sample_set, cfg in SAMPLE_SETS.items():
    mean, cov_inv = group_fits[cfg["group"]]
    dists, keep_mask = gate(aou, mean, cov_inv, cfg["threshold"], N_PCS)
    keep_masks[sample_set] = keep_mask

    keep_ids = aou.loc[keep_mask, "person_id"]
    out_path = os.path.join(FINAL_PCA_DIR, f"final_keep_ids_{sample_set}_{cfg['prob_tag']}.txt")
    keep_ids.to_csv(out_path, index=False, header=False)
    print(f"[{sample_set}] {keep_mask.sum()} / {len(aou)} kept -> {out_path}")

## `eur_uniform`: density-reweighted resample

Natural sampling density in PC space is highest near the centroid and thins toward the edges -- this can bias downstream heritability/GRM estimates toward the densest sub-cluster. Resamples `eur_loose`'s members so the result is closer to uniform across PC1-2 (restricted to 2 PCs -- density estimation in higher dimensions gets unreliable fast, and uniformity is only really meaningful/visualizable in low dimensions here).

Local density via k-th-nearest-neighbor distance (`cKDTree`, `O(N log N)` -- `gaussian_kde` is `O(N^2)` and won't scale to this N). Importance weight is inverse density (`distance^d`, `d=2`), then a weighted sample without replacement down to a target size.

In [ ]:
K_NEIGHBORS = 15          # for local density estimate
EUR_UNIFORM_TARGET_N = int(keep_masks["eur_base"].sum())   # same order of magnitude as eur_base, for comparability

eur_loose_df = aou.loc[keep_masks["eur_loose"]].reset_index(drop=True)
xy = eur_loose_df[["PC1", "PC2"]].values

tree = cKDTree(xy)
kth_dist, _ = tree.query(xy, k=K_NEIGHBORS + 1)   # +1: query includes the point itself at distance 0
kth_dist = kth_dist[:, -1]
kth_dist = np.maximum(kth_dist, np.finfo(float).eps)   # guard against exact duplicates

importance_weight = kth_dist ** 2   # d=2 (PC1, PC2) -- inverse density
importance_weight /= importance_weight.sum()

rng = np.random.default_rng(0)
n_target = min(EUR_UNIFORM_TARGET_N, len(eur_loose_df))
uniform_idx = rng.choice(len(eur_loose_df), size=n_target, replace=False, p=importance_weight)
eur_uniform_ids = eur_loose_df.loc[uniform_idx, "person_id"]

out_path = os.path.join(FINAL_PCA_DIR, "final_keep_ids_eur_uniform_uniform.txt")
eur_uniform_ids.to_csv(out_path, index=False, header=False)
print(f"[eur_uniform] {len(eur_uniform_ids)} / {len(eur_loose_df)} (from eur_loose) -> {out_path}")

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
axes[0].scatter(xy[:, 0], xy[:, 1], s=2, alpha=0.2, color="royalblue")
axes[0].set_title(f"eur_loose (n={len(eur_loose_df)})")
axes[1].scatter(xy[uniform_idx, 0], xy[uniform_idx, 1], s=2, alpha=0.2, color="darkorange")
axes[1].set_title(f"eur_uniform (n={len(uniform_idx)})")
for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.tight_layout()
plot_path = os.path.join(FINAL_PCA_DIR, "eur_uniform_vs_loose.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

## Orientation plot (not a filtering step)

AoU samples (downsampled for plotting speed) and `training_pca.tsv`'s reference points on the same PC1/PC2 axes -- both already share this exact PC space, so no projection is needed. Purely a visual check that each gate lands where expected relative to the reference clusters.

In [ ]:
PLOT_N_AOU = 50_000

plot_aou = aou.sample(n=min(PLOT_N_AOU, len(aou)), random_state=0)

fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(plot_aou["PC1"], plot_aou["PC2"], s=2, alpha=0.15, color="lightgray", label="AoU (all, downsampled)")

for sample_set in SAMPLE_SETS:
    mask = keep_masks[sample_set]
    sub = aou.loc[mask].sample(n=min(PLOT_N_AOU, mask.sum()), random_state=0)
    ax.scatter(sub["PC1"], sub["PC2"], s=3, alpha=0.4, label=sample_set)

for group in GROUPS:
    sub = ref[ref["pop_label"] == group]
    ax.scatter(sub["PC1"], sub["PC2"], s=20, marker="x", label=f"ref: {group}")

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("AoU samples vs. training_pca.tsv reference populations")
ax.legend(fontsize=7, markerscale=2, loc="best")
plt.tight_layout()
plot_path = os.path.join(FINAL_PCA_DIR, "orientation_plot.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {plot_path}")

## Next steps

`02_genome_wide_qc_thinning_batch_submit.ipynb` reads these keep-lists next, one `SAMPLE_SET` at a time, to build the GRM panel.